In [2]:
import torch
import torch.nn as nn
import math

In [18]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_k = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """
        Q: (batch, num_heads, seq_len_q, d_k)
        K: (batch, num_heads, seq_len_k, d_k)
        V: (batch, num_heads, seq_len_k, d_v)

        Return:
            context: (batch, num_heads, seq_len_q, dv)
            weights: (batch, num_heads, seq_len_q, seq_len_k)
        """
        d_k = Q.size(-1)
        
        scores = torch.matmul(Q, K.transpose(-1, -2)) # (batch, num_heads, seq_len_q, seq_len_k)
        scores = scores/math.sqrt(d_k)
        if mask != None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        weights = torch.softmax(scores, dim = -1)
        weights = self.dropout(weights)
        context = torch.matmul(weights, V) # (batch, num_heads, seq_len_q, d_v)
        return context, weights

    def forward(self, x, mask=None):
        # dimension: (batch, seq_len, d_model) -> (batch, seq_len, d_model)
        Q = self.W_Q(x)
        K = self.W_K(x)
        V = self.W_V(x)

        batch = Q.size(0)
        # (batch, seq_len, d_model) -> (batch, seq_len, num_heads, d_k) 
        # -> (batch, num_heads, seq_len, d_k)
        Q = Q.view(batch, -1, self.num_heads, self.d_k).transpose(1,2)
        K = K.view(batch, -1, self.num_heads, self.d_k).transpose(1,2)
        V = V.view(batch, -1, self.num_heads, self.d_k).transpose(1,2)

        attn_output, weights = self.scaled_dot_product_attention(Q, K, V, mask)
        # attn_output: (batch, num_heads, seq_len_q, d_v) -> (batch, seq_len, d_model)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch, -1, self.d_model)
        attn_output = self.W_O(attn_output)
        return attn_output

class ForwardFeedNetwork(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        result = self.linear2(
            self.dropout(
                self.relu(self.linear1(x))
            )
        )
        return result;
    
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len = 512, dropout = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000)) / d_model
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)
        
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.ffn = ForwardFeedNetwork(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # sub-layer-1: MultiHead Attention(Pre-Norm)
        normed = self.norm1(x)
        attn_output = self.attention(normed, mask)
        x = x + self.dropout1(attn_output)
        
        # sub-layer-2: FFN(Pre-Norm)
        normed = self.norm2(x)
        ffn_output = self.ffn(normed)
        x = x+ self.dropout2(ffn_output)

        return x

class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, num_classes, max_len = 512, dropout=0.1):
        super().__init__()

        # Embedding Layer
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_len)
        self.dropout = nn.Dropout(dropout)

        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])

        self.final_norm = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model, num_classes)
        self._init_weights()

    def _init_weights(self):
        nn.init.xavier_uniform_(self.token_embedding.weight)

    def forward(self, input_ids, mask = None):
        # input_ids: (batch, seq_len)
        # x dimension: (batch, seq_len, d_model)
        x = self.token_embedding(input_ids)
        x = self.positional_encoding(x)

        for layer in self.layers:
            x = layer(x, mask)

        x = self.final_norm(x)
        pooled = x.mean(dim=1)
        logits = self.classifier(pooled)
        return logits
        

    
        
        

In [19]:
model = TransformerEncoder(
    vocab_size=10000,
    d_model=128,
    num_heads=4,
    d_ff=512,
    num_layers=2,
    num_classes=2,
    max_len=256,
    dropout=0.1
)

# 打印参数量
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Total parameters: 1,677,058


In [20]:
dummy_input = torch.randint(0, 10000, (4, 32))
output = model(dummy_input)
print(f"Output shape: {output.shape}")  # expect (4, 2)

Output shape: torch.Size([4, 2])


In [25]:
def test_shapes():
    batch, seq_len, d_model, num_heads = 2, 10, 128, 4
    
    # Test MultiHeadAttention
    mha = MultiHeadAttention(d_model, num_heads)
    x = torch.randn(batch, seq_len, d_model)
    out = mha(x)
    assert out.shape == (batch, seq_len, d_model), f"MultiHeadAttention shape wrong: {out.shape}"
    print(f"MultiHeadAttention shape correct")

    # Test ForwardFeedNetwork
    ffn = ForwardFeedNetwork(d_model, 4*d_model)
    x = torch.randn(batch, seq_len, d_model)
    out = ffn(x)
    assert out.shape == (batch, seq_len, d_model), f"ForwardFeedNetwork shape wrong: {out.shape}"
    print(f"ForwardFeedNetwork shape correct")

    # Test EncoderLayer
    encoder_layer = EncoderLayer(d_model, num_heads, 4*d_model)
    x = torch.randn(batch, seq_len, d_model)
    out = encoder_layer(x)
    assert out.shape == (batch, seq_len, d_model), f"Encoder Layer shape wrong: {out.shape}"
    print(f"Encoder Layer shape correct")

    # Test Full TransformerEncoder
    model = TransformerEncoder(10000, d_model, num_heads, 4*d_model, 2, 2, max_len = 512, dropout=0.1)
    ids = torch.randint(0, 10000, (batch, seq_len))
    logits = model(ids)
    assert logits.shape == (batch, 2), f"TransformerEncoder shape wrong: {logits.shape}"
    print(f"TransformerEncoder shape correct")

test_shapes()

MultiHeadAttention shape correct
ForwardFeedNetwork shape correct
Encoder Layer shape correct
TransformerEncoder shape correct


In [30]:
def test_gradients():
    model = TransformerEncoder(1000, 128, 4, 512, 2, 2)
    x = torch.randint(0, 1000, (2, 10))
    labels = torch.tensor([0, 1])

    logits = model(x)
    loss = nn.CrossEntropyLoss()(logits, labels)
    loss.backward()

    for name, parameter in model.named_parameters():
        print(f"{name}")
        if parameter.requires_grad:
            assert parameter.grad is not None, f"No gradient for {name}"
            assert parameter.grad.abs().sum() > 0, f"Zero gradient for {name}"
    print(f"All parameters receiv non-zero gradient")

test_gradients()
    

token_embedding.weight
layers.0.attention.W_Q.weight
layers.0.attention.W_Q.bias
layers.0.attention.W_K.weight
layers.0.attention.W_K.bias
layers.0.attention.W_V.weight
layers.0.attention.W_V.bias
layers.0.attention.W_O.weight
layers.0.attention.W_O.bias
layers.0.ffn.linear1.weight
layers.0.ffn.linear1.bias
layers.0.ffn.linear2.weight
layers.0.ffn.linear2.bias
layers.0.norm1.weight
layers.0.norm1.bias
layers.0.norm2.weight
layers.0.norm2.bias
layers.1.attention.W_Q.weight
layers.1.attention.W_Q.bias
layers.1.attention.W_K.weight
layers.1.attention.W_K.bias
layers.1.attention.W_V.weight
layers.1.attention.W_V.bias
layers.1.attention.W_O.weight
layers.1.attention.W_O.bias
layers.1.ffn.linear1.weight
layers.1.ffn.linear1.bias
layers.1.ffn.linear2.weight
layers.1.ffn.linear2.bias
layers.1.norm1.weight
layers.1.norm1.bias
layers.1.norm2.weight
layers.1.norm2.bias
final_norm.weight
final_norm.bias
classifier.weight
classifier.bias
All parameters receiv non-zero gradient


In [31]:
def test_overfit():
    """能在小数据上 overfit = 模型实现没有大 bug"""
    model = TransformerEncoder(100, 64, 4, 256, 2, 2)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    # 造一个 tiny dataset: 8 个样本
    X = torch.randint(0, 100, (8, 16))
    y = torch.tensor([0, 1, 0, 1, 0, 1, 0, 1])

    model.train()
    for epoch in range(100):
        logits = model(X)
        loss = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 20 == 0:
            acc = (logits.argmax(1) == y).float().mean()
            print(f"Epoch {epoch+1}: "
                  f"loss={loss.item():.4f}, "
                  f"acc={acc.item():.2%}")

    final_acc = (model(X).argmax(1) == y).float().mean()
    assert final_acc > 0.9, \
        f"Failed to overfit! acc={final_acc:.2%}"
    print(f"\nOverfit success! acc={final_acc:.2%}")

test_overfit()

Epoch 20: loss=0.6375, acc=100.00%
Epoch 40: loss=0.0804, acc=100.00%
Epoch 60: loss=0.0015, acc=100.00%
Epoch 80: loss=0.0030, acc=100.00%
Epoch 100: loss=0.0027, acc=100.00%

Overfit success! acc=100.00%
